In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.utils import shuffle
import joblib

file_path = "C:/Users/Maria/Downloads/data_dirty.csv"
df = pd.read_csv(file_path)
df = shuffle(df, random_state=42).reset_index(drop=True)

output_folder = "C:/Users/Maria/Downloads/Project_Output"
os.makedirs(output_folder, exist_ok=True)

numeric_cols = df.select_dtypes(include=['int64','float64']).columns.tolist()
categorical_cols = df.select_dtypes(include=['object','category']).columns.tolist()

if numeric_cols:
    for col in numeric_cols:
        plt.figure()
        sns.histplot(df[col], kde=True)
        plt.title(f"{col} Histogram")
        plt.savefig(os.path.join(output_folder, f"{col}_histogram.png"))
        plt.close()

    for col in numeric_cols:
        plt.figure()
        sns.boxplot(x=df[col])
        plt.title(f"{col} Boxplot")
        plt.savefig(os.path.join(output_folder, f"{col}_boxplot.png"))
        plt.close()

    if len(numeric_cols) > 1:
        plt.figure(figsize=(10,8))
        sns.heatmap(df[numeric_cols].corr(), annot=True, cmap="coolwarm")
        plt.title("Correlation Heatmap")
        plt.savefig(os.path.join(output_folder, "correlation_heatmap.png"))
        plt.close()

        sns.pairplot(df[numeric_cols])
        plt.savefig(os.path.join(output_folder, "pairplot.png"))
        plt.close()

for col in categorical_cols:
    plt.figure()
    sns.countplot(x=df[col])
    plt.title(f"{col} Countplot")
    plt.savefig(os.path.join(output_folder, f"{col}_countplot.png"))
    plt.close()

df = df.drop_duplicates()
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.strip().str.lower()
for col in df.select_dtypes(include=['int64','float64']).columns:
    df[col].fillna(df[col].median(), inplace=True)
for col in df.select_dtypes(include='object').columns:
    df[col].fillna(df[col].mode()[0], inplace=True)
for col in df.select_dtypes(include=['int64','float64']).columns:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5*IQR
    upper = Q3 + 1.5*IQR
    df = df[(df[col]>=lower) & (df[col]<=upper)]

X = df.iloc[:, :-1]
y = df.iloc[:, -1]
numeric_cols = X.select_dtypes(include=['int64','float64']).columns.tolist()
categorical_cols = X.select_dtypes(include=['object','category']).columns.tolist()

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])
preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_cols),
    ('cat', categorical_transformer, categorical_cols)
])
pipeline = Pipeline([('preprocessor', preprocessor)])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
pipeline.fit(X_train)
X_train_transformed = pipeline.transform(X_train)
X_test_transformed = pipeline.transform(X_test)

df.to_csv(os.path.join(output_folder,"cleaned_data.csv"), index=False)
joblib.dump(pipeline, os.path.join(output_folder,"preprocessing_pipeline.pkl"))

with open(os.path.join(output_folder,"README.txt"), "w") as f:
    f.write("Dataset: Titanic / Dirty Sample Dataset\n")
    f.write(f"Original shape: {df.shape[0]} rows, {df.shape[1]} columns\n")
    f.write("EDA Key Findings:\n")
    for col in numeric_cols:
        f.write(f"{col}: mean={df[col].mean():.2f}, median={df[col].median():.2f}, std={df[col].std():.2f}\n")
    for col in categorical_cols:
        f.write(f"{col} value counts:\n{df[col].value_counts().to_dict()}\n")
    f.write("\nPreprocessing Steps:\n")
    f.write("- Duplicates removed\n")
    f.write("- Text columns standardized to lowercase\n")
    f.write("- Missing numeric values filled with median\n")
    f.write("- Missing categorical values filled with mode\n")
    f.write("- Outliers removed using IQR\n")
    f.write("- Numeric features scaled with StandardScaler\n")
    f.write("- Categorical features one-hot encoded\n")
    f.write("- Train/test split performed\n")
    f.write("- Preprocessing pipeline saved as preprocessing_pipeline.pkl\n")

print("Project complete. All outputs saved in:", output_folder)

Project complete. All outputs saved in: C:/Users/Maria/Downloads/Project_Output
